Tukaj bo izvajane vse analize in filtrirane kode za vzorce, ki jih iščemo; medtem pa bo pr.py bil pomozna knjiznica za osnove funkcije kot so nalaganje tsv-jev in šiščenje podatkovnih zbirk.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from data_utils import *
import sklearn

asdasd


In [ ]:
df_ren = share_of_renewables()
df_ren = add_country_names(df_ren)

df_filtered = df_ren[
    (df_ren['nrg_bal'] == 'REN') & 
    (df_ren['year'] >= 2014) &
    (df_ren['geo'].str.len() == 2)
]

pivot_df = df_filtered[df_filtered['nrg_bal'] == 'REN'].pivot(index='drzava', columns='year', values='value')
pivot_df = pivot_df.interpolate(axis=1)
pivot_df = pivot_df.sort_values(by=pivot_df.columns[-1], ascending=False)

plt.figure(figsize=(14, 10))
sns.heatmap(pivot_df, 
            annot=True,
            fmt=".1f",
            cmap="RdYlGn",
            linewidths=.5, 
            cbar_kws={'label': 'Delež OVE (%)'})

plt.title('Delež obnovljivih virov energije po državah (2014–danes)', fontsize=16)
plt.xlabel('Leto')
plt.ylabel('Država (Koda)')
plt.show()

In [ ]:
from sklearn.linear_model import LinearRegression
import matplotlib.patches as mpatches

results = []

for country in pivot_df.index:
    temp_series = pivot_df.loc[country].dropna()
    if len(temp_series) < 2:
        print(f"Preskoči {country}: Ni dovolj podatkov.")
        continue
        
    y = temp_series.values
    X = np.array(temp_series.index).reshape(-1, 1)
    
    model = LinearRegression()
    model.fit(X, y)
    pred_2030 = model.predict([[2030]])[0]
    
    results.append({
        'drzava': country, 
        'zadnje_leto': y[-1], 
        'napoved_2030': pred_2030
    })


df_pred = pd.DataFrame(results)
df_pred['dosega_cilj'] = df_pred['napoved_2030'] >= 42.5
df_pred = df_pred.sort_values(by='napoved_2030', ascending=False)
df_viz = df_pred.sort_values(by='napoved_2030', ascending=True)

colors = ['#2ca02c' if x else '#d62728' for x in df_viz['dosega_cilj']]

plt.figure(figsize=(14, 12))

bars = plt.barh(df_viz['drzava'], df_viz['napoved_2030'], color=colors, edgecolor='black', alpha=0.8)

plt.axvline(x=42.5, color='black', linestyle='--', linewidth=2.5, label='EU Cilj 2030 (42.5%)')
plt.title('Napoved deleža OVE za leto 2030: Doseganje cilja 42,5 %', fontsize=18, fontweight='bold')
plt.xlabel('Napovedan delež OVE (%)', fontsize=14)
plt.ylabel('Država', fontsize=14)
plt.grid(axis='x', linestyle='-', alpha=0.3)

red_patch = mpatches.Patch(color='#d62728', label='Cilj NE bo dosežen')
green_patch = mpatches.Patch(color='#2ca02c', label='Cilj BO dosežen')
plt.legend(handles=[green_patch, red_patch, plt.gca().get_lines()[0]], loc='lower right', fontsize=12)

plt.tight_layout()
plt.show()

print(df_pred[['drzava', 'zadnje_leto', 'napoved_2030', 'dosega_cilj']])